In [1]:
import pandas as pd
import numpy as np
import os
from openpyxl import load_workbook
import seaborn as sns
import matplotlib.pyplot as plt

### Batch process all the responses to merge into a singular dataframe

#### Folder filtering

In [3]:
ignore = ['.ipynb_checkpoints','MSA 2420','MSA 2451','MSA 2482','MSA 2544','MSA 2669'] # Folders to ignore (skip unrelated folders)

folders =[folder for folder in filter(os.path.isdir, os.listdir(os.getcwd())) if folder not in ignore]
folders = [int(folder) for folder in folders]

# If some specific folders are desired :
specific = None
specific = [19,27,39,59,92,149,249,427,753,1367,2548,4879,9585,19316,39909,84503]

if specific != None :
    folders = [folder for folder in folders if folder in specific ]
    
# Sort the list in increasing order
folders.sort()

print(folders)

[19, 27, 39, 59, 92, 149, 249, 427, 753, 1367, 2548, 4879, 9585, 19316, 39909, 84503]


#### Identify available csv files

In [4]:
fp_dict = {}

for folder in folders :
    filepaths = []
    folder_dir = os.path.join(os.getcwd(),str(folder))
    folder_files = [x for x in os.listdir(folder_dir) if os.path.splitext(x)[1] == '.csv']
    
    # This will skip empty directories (as for loops will not iterate empty variables)
    for file in folder_files :
        file_complete_path = os.path.join(os.getcwd(),str(folder),file)
        filepaths.append(file_complete_path)
    # Add a dictionary entry if the folder contains any csv file : 
    if len(filepaths) != 0 : 
        fp_dict[folder] = filepaths

##### Extract data from the folders

In [5]:
data = pd.DataFrame()
corr_matrices = {}


# Identify the EDPs to look out for :
params_to_keep = ['PFA','PFD','PID']

accel_columns = ['PFA']

# Mark the collapse limit 
MAX_PID = 0.05


for folder in fp_dict.keys() :
    # Note : this assumes only 1 file per directory
    fp = fp_dict[folder][0] 
    
    df = pd.read_csv(fp,index_col = 0)

    # First, lets set acceleration units relative to "g" : 
    g = 386.4 # inches/s**2 
    
    accel_cols = [x for x in df.columns if any(x for y in accel_columns if y in x)]
    df[accel_cols] = df[accel_cols]/g    
    
    
    # Second, clear collapse results 
    cols_keep = [x for x in df.columns if 'PID' in x]
    PID_df = df[cols_keep]    
    Non_collapse = PID_df[~(PID_df > MAX_PID).any(axis=1)]

    df = df.loc[Non_collapse.index,:]

    # Third, clear the unecessary columns and extract statistics :
    
        # Single out the columns to keep :
    cols_to_keep = [x for x in df.columns if any(x for y in params_to_keep if y in x)]     
     
    selected_cols = df[cols_to_keep] 
    
    # Derive correlations between sampled parameters
    corr_matrices[folder] = selected_cols.corr(method = 'spearman')

    
        # Edit the results to keep median and standard deviation of the natural logarithm
    median = selected_cols.describe().iloc[[-3],:].T    

    log_df = np.log(selected_cols)
    std_log = pd.DataFrame(np.std(log_df))

        # Re-assemble everything :
    selected_stats = pd.concat([median,std_log],axis =1 )    
    
        # Relabel the columns
    selected_stats.columns = ['median','std_log']
    
        # Relabel the index to account for MRIs : 
    index = list(selected_stats.index)
    MRI_index = [str(folder)+x[1:] for x in index]
    selected_stats.index = MRI_index    
    
    data = pd.concat([data,selected_stats])
    
display(data)

,median,std_log
19-PFA-1-1,0.002810,0.332405
19-PFD-1-1,0.167011,0.104277
19-PID-1-1,0.000535,0.104277
19-PFA-2-1,0.003814,0.331601
19-PFD-2-1,0.249283,0.104854
...,...,...
84503-PFD-18-1,50.774100,0.246089
84503-PID-18-1,0.009528,0.156417
84503-PFA-19-1,0.223142,0.075782
84503-PFD-19-1,51.695700,0.241011


#### Export the resulting data

###### EDPs

In [6]:
fname = 'building_EDPs.xlsx'

if os.path.isfile(fname): # The file already exists
    with pd.ExcelWriter(fname) as writer : 
        data.to_excel(writer,sheet_name = 'demand_data')

else : 
    with pd.ExcelWriter(fname) as writer : 
        data.to_excel(writer,sheet_name = 'demand_data')
        writer.handles = None

###### Correlations

In [7]:
fname = 'building_EDPs_correlations.xlsx'

# Adjust column and index tags
for folder in fp_dict.keys() :
    indices  = [x[2:] for x in list(corr_matrices[folder])]
    corr_matrices[folder].index = indices
    corr_matrices[folder].columns = indices

if os.path.isfile(fname) :  # The file already exists
    with pd.ExcelWriter(fname) as writer :
        for folder in fp_dict.keys() : 
            corr_matrices[folder].to_excel(writer, sheet_name = f"{folder}")

else : 
    with pd.ExcelWriter(fname) as writer : 
        for folder in fp_dict.keys() : 
            corr_matrices[folder].to_excel(writer, sheet_name = f"{folder}")
        writer.handles = None